# Notebook 34 — Building an MCP Server for an HF Agent

    ## Learning objectives

    - Distinguish MCP tools, resources, prompts, and transports
- Create and inspect a FastMCP server from a notebook
- Bridge discovered MCP tools into a Hugging Face agent safely

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['mcp>=1.6']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 34.1 MCP standardizes context integration

An MCP host connects to servers through clients. Servers expose tools (actions),
resources (readable context), and prompts (templates). The protocol standardizes
discovery and invocation; it does not automatically make a server trusted. Hosts still
need permission policy, schema validation, isolation, and user-visible approvals.


In [ ]:
# This cell writes a small course artifact, not credentials.
from pathlib import Path
server_source = """from mcp.server.fastmcp import FastMCP
from pathlib import Path

mcp = FastMCP("course-notes")

@mcp.tool()
def word_count(text: str) -> dict:
    # Count whitespace-separated words in bounded text.
    if len(text) > 10_000:
        raise ValueError("text too large")
    return {"words": len(text.split())}

@mcp.resource("course://syllabus")
def syllabus() -> str:
    # Return a short course description.
    return "Decoder internals, training, retrieval, agents, evaluation, and serving."

if __name__ == "__main__":
    mcp.run()
"""
Path("demo_mcp_server.py").write_text(server_source)
print("wrote demo_mcp_server.py")


In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def inspect_server():
    params = StdioServerParameters(command=sys.executable, args=["demo_mcp_server.py"])
    async with stdio_client(params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            print("tools:", [t.name for t in tools.tools])
            print("resources:", [str(r.uri) for r in resources.resources])
            result = await session.call_tool("word_count", {"text": "MCP keeps integrations composable"})
            print(result.content[0].text)

# Jupyter already runs an event loop, so use top-level await—not asyncio.run().
await inspect_server()


## 34.2 Bridge deliberately

Convert MCP schemas to the chat model's tool format, but keep execution in the host.
Maintain an allowlist by server/tool, validate every argument again, cap response sizes,
sanitize errors, and treat resource/tool contents as untrusted. Network transports need
authentication and origin controls; local stdio servers inherit process privileges.


## 34.3 Full capability surface and lifecycle

Initialization negotiates a protocol version and capabilities before normal operation. Servers may expose tools,
resources and templates, prompts, logging, completion suggestions, notifications, and experimental tasks. Clients
may expose roots, sampling, elicitation, and task support. Both sides must use only negotiated capabilities and
should tolerate pagination and list-change notifications. Shutdown uses transport semantics.

Tools represent model-controlled actions. Resources are application-controlled context identified by URIs. Prompts
are user-selectable templates rather than hidden system policy. Roots communicate filesystem scope but do not grant
authorization by themselves. Sampling lets a server request model generation through the client without receiving
the user's model credential; the client retains model choice, review, and approval responsibility.


In [ ]:
capability_contract = {
    "server": ["tools", "resources", "prompts", "logging", "completions", "tasks"],
    "client": ["roots", "sampling", "sampling.tools", "elicitation", "tasks"],
}
for side, capabilities in capability_contract.items(): print(side, "->", ", ".join(capabilities))


## 34.4 Transports, authorization, and identity

Stdio is appropriate for a host-launched local subprocess. Credentials come from its environment and should be
narrowed to that server; stdout belongs to protocol traffic and diagnostics go to stderr. Streamable HTTP requires
authenticated transport design, Origin validation, protocol-version headers, session/resumption handling according
to the negotiated specification version, and bounded request/stream resources.

HTTP authorization is separate from tool arguments. Verify user and client identity server-side, request incremental
scopes, prevent token passthrough and confused-deputy behavior, and bind authorization to the actual resource. Never
trust a model claim such as “the user approved.” Elicitation UI must identify the requesting server, permit refusal,
validate returned fields, and never request passwords or API keys through form-mode elicitation.


In [ ]:
policy = {
    ("course-notes", "word_count"): {"effect":"read", "approval":False, "max_calls":20},
    ("course-notes", "publish_notes"): {"effect":"external_write", "approval":True, "max_calls":1},
}
requested = ("course-notes", "publish_notes")
print("host decision:", policy.get(requested, {"deny":True}))


## 34.5 Sampling, elicitation, tasks, and untrusted content

Sampling requests may include tool choices when the client advertises support. Display or policy-check the proposed
prompt, cap tokens, restrict tools, and avoid automatically injecting context from unrelated servers. Elicitation
collects non-sensitive structured information or directs a user through a URL flow; bind completion to the initiating
user to prevent phishing. Experimental tasks support polling, cancellation, and deferred retrieval for long work;
version-gate them because the surface is evolving.

Everything returned by a server is untrusted: resource text can inject prompts, icons can contain active content,
tool results can be oversized, and schemas or descriptions can manipulate selection. Isolate each server connection,
namespace capabilities, cap payloads, sanitize rendered content, disallow credential forwarding, and prevent one
server from learning another server's resources unless the host explicitly permits it.


In [ ]:
def approve_sampling(request, allowed_tools=frozenset({"word_count"}), max_tokens=512):
    proposed = {tool["name"] for tool in request.get("tools", [])}
    return {"approved": proposed <= allowed_tools and request.get("maxTokens", 0) <= max_tokens,
            "tools": sorted(proposed), "bounded_tokens": min(request.get("maxTokens", 0), max_tokens)}
print(approve_sampling({"maxTokens":200, "tools":[{"name":"word_count"}]}))
print(approve_sampling({"maxTokens":5000, "tools":[{"name":"shell"}]}))


## 34.3 Protocol lifecycle and capability negotiation

An MCP client connects over a transport, initializes a session, exchanges protocol versions and
capabilities, then discovers primitives. Tools are model-invocable operations. Resources are
application-controlled readable data identified by URIs and may support templates/subscriptions.
Prompts are reusable templates the user/host can select. Servers can request sampling from the
host under negotiated capability and policy. Notifications communicate changes without a request.

Transports change deployment/security. Stdio is simple and inherits the spawned process's local
privileges/environment. HTTP-based remote transport introduces network identity, TLS, origins,
authorization, session handling, and exposure to the internet. The protocol describes messages;
it does not decide which server is trusted or which user may invoke which capability.


In [ ]:
# Inspect complete discovered schemas rather than only names.
async def describe_server():
    params = StdioServerParameters(command=sys.executable, args=["demo_mcp_server.py"])
    async with stdio_client(params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            response = await session.list_tools()
            for tool in response.tools:
                print({"name": tool.name, "description": tool.description,
                       "inputSchema": tool.inputSchema})
await describe_server()


## 34.4 Server engineering and error design

Tools should be thin adapters over tested domain functions, with typed parameters, bounded
inputs/outputs, explicit side-effect annotations in descriptions, and sanitized errors. Avoid
ambient access to an entire filesystem or network. Resolve paths against an allowed root and
defend against symlinks/traversal. Inject scoped credentials per request/user rather than giving
a long-lived universal token to the process. Apply timeouts and cancellation to downstream work.

Resources need stable URI schemes, MIME types, provenance, size limits, and authorization.
Resource contents are untrusted when inserted into an LLM prompt. Prompt templates should be
versioned and transparent. Log protocol/tool metadata while redacting secrets and sensitive
content. Test the server functions directly, then through an MCP client, then through a model
host; these layers catch different failures.


In [ ]:
# Directly test the generated server's domain function without an LLM.
import importlib.util
spec = importlib.util.spec_from_file_location("demo_mcp_server", "demo_mcp_server.py")
module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
for text in ["one two three", "", "word " * 50]:
    print(repr(text[:20]), module.word_count(text))
try:
    module.word_count("x" * 10_001)
except Exception as exc:
    print("bounded input test:", type(exc).__name__, str(exc))


## 34.5 Host bridge and security policy

The host decides what the model sees and what is executed. Convert discovered schemas to the
model's tool format, but maintain a server/tool allowlist, validate arguments locally, bind calls
to the current user, and authorize every operation. Cap tool result bytes/tokens and label their
origin. Do not automatically install/connect servers suggested by untrusted content. Surface the
exact server identity and effect when requesting approval.

Threats include malicious servers, compromised dependencies, tool-name collisions, schema
deception, indirect prompt injection in resources/results, credential exfiltration, confused-
deputy actions, cross-tenant access, and denial of service. For local servers, review command,
arguments, environment variables, binary path, working directory, and filesystem/network scope.
For remote servers, verify endpoint/authentication and constrain redirects/origins.

**Deployment reference:** pin SDK/server versions; use a lockfile; isolate process/container;
least-privilege credentials; authenticate users and servers; tool/resource policy; bounded
payloads/time; approvals; audit trail; health/restart policy; protocol compatibility tests; and
an incident path to revoke the connection.


## 34.6 MCP reference

| Primitive | Meaning |
|---|---|
| Tool | Callable operation with input schema |
| Resource | URI-addressed readable content/context |
| Resource template | Parameterized resource URI pattern |
| Prompt | Discoverable reusable prompt template |
| Sampling | Server request for host-mediated model generation |
| Capability | Negotiated feature support during initialization |
| Transport | Message channel such as stdio or HTTP-based remote transport |

MCP standardizes exchange, not trust. The host remains responsible for server identity, installation/
connection approval, user authorization, tool policy, input validation, result size, untrusted content,
credential scoping, and audit. Stdio processes can access what their OS identity can access. Remote
servers add authentication, TLS/origin/redirect, tenancy, and availability concerns.

Test domain functions, server protocol discovery/invocation, host schema conversion, model behavior,
and security policy separately. Pin protocol/SDK/server versions and test compatibility. Avoid broad
filesystem/shell/browser tools unless strongly isolated and explicitly authorized.


## 34.10 MCP capability discovery is not authorization

Tools, resources, and prompts advertised by a server describe capabilities; they do not grant a user permission to use them. The host must authenticate servers, map user identity and scopes, validate schemas, and authorize every call. Treat descriptions and returned content as untrusted. Pin or approve server versions and expose only needed capabilities to each agent. A compromised server should not be able to instruct the host to call another privileged capability.


In [ ]:
capability={"server":"catalog","tool":"lookup","declared":True}; principal={"scopes":{"catalog:read"}}; required="catalog:read"
authorized=capability["declared"] and required in principal["scopes"]; print("authorized",authorized)


## 34.11 Protocol failure and lifecycle tests

Exercise initialization/version negotiation, capability changes, cancellation, concurrent request IDs, progress, malformed messages, oversized results, disconnects, restart, and server timeouts. Correlate requests and responses without trusting order. Normalize protocol failures into host-level error categories and avoid replaying non-idempotent effects. Integration tests should use a fake transport and deterministic server before subprocess or network transports. Log metadata and hashes while redacting arguments or results that may contain secrets.


In [ ]:
pending={1:"list_tools",2:"call_tool"}; responses=[{"id":2,"result":"receipt-7"},{"id":1,"result":["lookup"]}]; completed={}
for response in responses: completed[pending.pop(response["id"])]=response["result"]
print(completed,"pending",pending)


## Exercises

    1. Add a parameterized resource and read it through the client.
2. Write a host policy that permits reads but requires approval for writes.
3. Threat-model a malicious MCP server returning prompt-injection text.
4. Add a prompt, root, paginated list, and mocked sampling request with capability checks.
5. Design Streamable HTTP authorization and elicitation flows without token passthrough.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
